# Exercises XP – Week 7, Day 1: The Rise of LLMs
Run in **Google Colab** (a GPU is optional; GPT-2 is small).

## 🌟 Exercise 1: What are Large Language Models (LLMs)?
**In my own words:** a Large Language Model is a very large neural network, usually a **Transformer** with billions of parameters, trained on huge amounts of text (books, websites, code) to do one simple-sounding task: **predict the next piece of text**. By learning to predict the next word in billions of examples, it picks up grammar, facts, writing styles and even some reasoning patterns.

They're designed to **understand and generate human language**: answering questions, summarising documents, translating, writing text or code, classifying sentiment, and holding conversations. The same pre-trained model can be adapted to many different tasks, with fine-tuning or just with the right instructions (prompts).

In [ ]:
# Install necessary libraries
!pip install transformers matplotlib --quiet

# Import required libraries
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# 2. Loading a pretrained model and tokenizer
model_name = "gpt2"      # GPT-2 is used here for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print(f"\nModel '{model_name}' loaded successfully!")
print("""
GPT-2 is a causal language model, meaning it predicts the next word in a sequence.
It has been trained on a diverse dataset and can generate coherent, contextually relevant text.
""")

## 🌟 Exercise 2: Transformer Architecture and Tokenization
**Tokenization** is the first step before a model can read text. A model can't work with letters or words directly, only with numbers, so the **tokenizer** splits the text into small pieces called **tokens** and replaces each one with its **ID** in the model's vocabulary (GPT-2 knows ~50,000 tokens). Tokens are often whole common words ("the", "cat"), but rare or long words are broken into sub-word pieces ("tokenization" → "token" + "ization"). This way the model can handle any word, even ones it has never seen, with a fixed-size vocabulary.

In [ ]:
text = "Psychologists are learning how language models understand emotions."

# 2. Tokenize input text
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print(f"Original Text: {text}")
print(f"Tokens: {tokens}")
print(f"Token IDs: {token_ids}")

# 3. Visualizing the tokenization process
plt.figure(figsize=(12, 4))
plt.bar(range(len(tokens)), token_ids, color="skyblue")
plt.xticks(range(len(tokens)), tokens, rotation=45, ha="right")
plt.xlabel("Token")
plt.ylabel("Token ID in GPT-2's vocabulary")
plt.title("How GPT-2 tokenizes the sentence")
plt.tight_layout()
plt.show()

## 🌟 Exercise 3: Understanding Token IDs and Special Prefixes

In [ ]:
for token, token_id in zip(tokens, token_ids):
    print(f"{token!r:>18} → {token_id}")

# Decoding the IDs gives back the original text
print("\nDecoded:", tokenizer.decode(token_ids))

**What does the prefix "Ġ" mean?** GPT-2 uses *byte-level BPE*, where the **space before a word becomes part of the token**, and it's displayed as **Ġ**. So `"Ġare"` means **" are"** (with a leading space), while `"are"` without Ġ would be the same letters appearing **inside or at the start of** a word. That's why the first word of the sentence has no Ġ, but most later words do. It lets the tokenizer rebuild the exact original text, spaces included, when decoding. Pieces of a split word (e.g. `"ists"` after `"Psych"`/`"olog"`) also have no Ġ, because there's no space before them.

## 🌟 Exercise 4: Pretraining vs. Fine-Tuning
- **Pretraining** is the first, huge training phase. The model reads an enormous amount of general text (much of the internet, books, code) and learns by predicting the next token (GPT) or filling in hidden words (BERT). No human labels are needed: the text itself is the answer key. This phase is extremely expensive (weeks on thousands of GPUs), and it gives the model its general knowledge of language and the world.
- **Fine-tuning** is a second, much smaller phase. We take the pre-trained model and continue training it on a **specific, usually labelled dataset** for a particular task or style, for example classifying customer reviews, answering medical questions, or following instructions politely as a chat assistant. Because the model already "knows" language, fine-tuning needs far less data and computing power, and it can be done in hours.

In short: **pretraining builds general understanding; fine-tuning specialises it.**

## 🌟 Exercise 5: Generate Simple Text

In [ ]:
input_text = "The most important skill for a data scientist is"

input_ids = tokenizer(input_text, return_tensors="pt").input_ids
output_ids = model.generate(
    input_ids,
    max_new_tokens=40,          # how many new tokens to add
    do_sample=True,             # sample instead of always picking the most likely token
    temperature=0.8,            # a bit of creativity
    top_k=50,
    pad_token_id=tokenizer.eos_token_id,
)
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f"Input: {input_text}")
print(f"Generated Output: {output_text}")

GPT-2 continues the prompt **one token at a time**: at each step it predicts a probability for every possible next token, picks one, appends it, and repeats. The result is fluent but not always factual or on-topic. GPT-2 is a small, older model (124M parameters) with no instruction tuning.